In [ ]:
import torch
import numpy as np
from typing import Optional, Tuple, Dict, Any
from dataclasses import dataclass
import time
import gc
import os

@dataclass
class PolySBMConfig:
    num_steps: int = 1000
    num_replicas: int = 100
    dt: float = 0.5
    a0: float = 1.0
    use_ternary: bool = True
    threshold_scale: float = 0.7
    wall_position: float = 1.0
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'


class PolynomialSimulatedBifurcationMachine:
    def __init__(self, 
                 J: Optional[np.ndarray] = None,
                 T: Optional[np.ndarray] = None,
                 h: Optional[np.ndarray] = None,
                 device: str = 'cpu',):
        
        self.device = torch.device(device)
        self.dtype = torch.float32

        self.N = T.shape[0]
        
        
        self.J = torch.tensor(J, dtype=self.dtype, device=self.device)

        self.h = torch.tensor(h, dtype=self.dtype, device=self.device)

        self.T = torch.tensor(T, dtype=self.dtype, device=self.device)
        
        # Нормализация
        self.sigma = np.std(T.flatten()) if np.std(T.flatten()) > 0 else 1.0
        
    
    def _compute_local_field(self, sign_q: torch.Tensor) -> torch.Tensor:
       
        field_pair = torch.mm(sign_q, self.J.T)
        
        field_triple = torch.einsum('ijk,rj,rk->ri', self.T, sign_q, sign_q)
        
        return field_pair + field_triple + self.h
    
    def _compute_energy(self, spins: torch.Tensor) -> torch.Tensor:
        
        energy = -torch.einsum('ij,ri,rj->r', self.J, spins, spins)
        energy -= torch.einsum('ijk,ri,rj,rk->r', self.T, spins, spins, spins)
        energy -= torch.einsum('ri,i->r', spins, self.h)
        
        return energy
    
    def solve(self, config: PolySBMConfig, verbose: bool = False):

        q = torch.rand(config.num_replicas, self.N, device=self.device, dtype=self.dtype) * 0.2 - 0.1
        p = torch.rand(config.num_replicas, self.N, device=self.device, dtype=self.dtype) * 0.2 - 0.1
        
        c0 = 0.7 * config.a0 / (self.sigma * np.sqrt(self.N))
        c0 = torch.tensor(c0, device=self.device, dtype=self.dtype)
        
        best_energy = float('inf')
        best_spins = None
        data_spins = []
        
        for step in range(config.num_steps):
            t = step / config.num_steps
            a_t = config.a0 * t
            

            if config.use_ternary:
                delta_t = config.threshold_scale * t
                sign_q = torch.zeros_like(q)
                mask = torch.abs(q) > delta_t
                sign_q[mask] = torch.sign(q[mask])
            else:
                sign_q = torch.sign(q)
            
            # Поле
            field = self._compute_local_field(sign_q)
            

            p -= ((config.a0 - a_t) * q - c0 * field) * config.dt
            q += config.a0 * p * config.dt
            
            # Стенка
            wall_mask = torch.abs(q) > config.wall_position
            q_sign = torch.sign(q)
            q = torch.where(wall_mask, torch.sign(q), q)
            p = torch.where(wall_mask, torch.tensor(0.0, device=self.device), p)
            
            
#             # Мониторинг
#             if step % 10 == 0:
#                 spins = torch.sign(q)
#                 data_spins.append(spins.cpu())
#                 energy = self._compute_energy(spins)
#                 min_e = energy.min().item()
                
#                 if min_e < best_energy:
#                     best_energy = min_e
#                     best_idx = energy.argmin().item()
#                     best_spins = spins[best_idx].cpu().numpy()
                
                
                
#                 if verbose:
#                     print(f"Шаг {step}: E = {best_energy:.4f}")
        
        
        torch.cuda.empty_cache()
        
        return None#best_spins, data_spins

In [ ]:
def compute_energy_sparse(x, H1, H2, H3):

    energy = 0.0
    n = len(x)
    
    # Линейные
    energy += np.dot(x, H1)
    
    # Квадратичные (только верхний треугольник)
    for i in range(n):
        for j in range(i+1, n):
            if H2[i, j] != 0:
                energy += H2[i, j] * x[i] * x[j]
    
    # Кубические (только i < j < k)
    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if H3[i, j, k] != 0:
                    energy += H3[i, j, k] * x[i] * x[j] * x[k]
    
    return energy

In [ ]:
for i in range(20, 50):
    torch.cuda.empty_cache()
    gc.collect()
    H1=np.zeros(200) 
    H2=np.zeros((200, 200)) 
    H3=np.load(f"J_T_h_1/T_{i}.npy")
       
    num_replicas = 5000
    num_steps = 500
    config = PolySBMConfig(
            num_steps=num_steps,
            num_replicas=num_replicas,
            dt=0.05,
            a0=0.75,
            use_ternary=True,
            device='cuda',


        )

    solver = PolynomialSimulatedBifurcationMachine(
        J=-H2, T=-H3, h=-H1,
        device='cuda'
    )
    spins, data = solver.solve(config, verbose=False)
    print(i, compute_energy_sparse(spins[:200], H1, H2, H3))
    np.save(f"solution_data_1/polySBM_{i}.npy", data)